## Stage 5 — STL Export

Converts the SIMP density field to a Fusion-360-quality watertight STL.

**Rust voxel path** (USE_RUST_SOLVER=True): reads `base_part_density.npy` (nz×ny×nx f32), runs skimage marching cubes, trimesh repair + Taubin smoothing + quadric decimation.

**FEniCSx tet path**: extracts the isosurface boundary from the DG0 density on the tet mesh.

Cell 0 — Parameters (tag: parameters)

In [1]:
# Cell 0 — tagged: parameters
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")

PART_NAME_OVERRIDE = None   # set by pipeline_full.ipynb in sweep mode
STAGE04_HANDOFF    = "outputs/meshes/cantilever_arm_stage04.json"

# ── Isosurface threshold ─────────────────────────────────────────────────
# Start with 0.3 after well-polarized two-stage run. Raise if mesh is bloated.
DENSITY_THRESHOLD  = 0.45

# ── Smoothing ─────────────────────────────────────────────────────────────
SMOOTH_ITERATIONS  = 10     # Taubin passes (0 = raw MC, 10 = light, 30 = smooth)
TAUBIN_LAMB        = 0.5
TAUBIN_NU          = -0.53  # negative in standard Taubin formulation

# ── Simplification ────────────────────────────────────────────────────────
TARGET_FACE_FRACTION = 0.30  # keep 10% of raw MC faces (target ~10k-50k)
MIN_FACES            = 5_000 # floor regardless of fraction

# ── Island removal ────────────────────────────────────────────────────────
REMOVE_ISLANDS     = True
ISLAND_VOL_FRAC    = 0.01   # drop components < 1% of total volume

RENDER_PLOTS = False


Cell 1 — Load Stage 4 handoff

In [2]:
# Cell 1 — Resolve paths from stage04 handoff
from pathlib import Path
import json
import numpy as np

# ── Locate Stage 4 handoff ────────────────────────────────────────────────
if STAGE04_HANDOFF is not None:
    handoff_path = Path(STAGE04_HANDOFF)
else:
    # Auto-detect: find most recent *_stage04.json
    candidates = sorted(Path("outputs/meshes").glob("*_stage04.json"))
    assert candidates, \
        "No *_stage04.json found in outputs/meshes/ — run notebook 04 first."
    handoff_path = candidates[-1]

assert handoff_path.exists(), f"Stage04 handoff not found: {handoff_path}"
handoff = json.loads(handoff_path.read_text())

# ── Extract metadata ──────────────────────────────────────────────────────
solver_type = handoff.get("solver", "rust_voxel")
grid_config = handoff["grid"]          # KeyError is intentional: grid is required
part_name   = PART_NAME_OVERRIDE if PART_NAME_OVERRIDE else handoff["part_name"]

print(f"Stage04 handoff: {handoff_path.name}")
print(f"  Part:      {part_name}")
print(f"  Solver:    {solver_type}")
print(f"  Grid:      {grid_config['nx']}×{grid_config['ny']}×{grid_config['nz']}")
print(f"  Converged: {handoff.get('converged')}  "
      f"Iters: {handoff.get('n_iterations')}  "
      f"C_final: {handoff.get('final_compliance', 0):.4e}")

Stage04 handoff: motor_mount_stage04.json
  Part:      motor_mount
  Solver:    rust_voxel
  Grid:      70×60×80
  Converged: True  Iters: 235  C_final: 5.8807e-02


Cell 2 — Load density field and inspect polarization

In [3]:
# Cell 2 — Load density + checkpoint
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: inspect before proceeding.
# A good two-stage run should show a strongly bimodal histogram.
# If you see a broad unimodal hump, re-run notebook 04 with the two-stage
# continuation (Stage1: p=2 → Stage2: p=3 warm-started from Stage1).
# ──────────────────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt

if solver_type == "rust_voxel":
    nx = grid_config["nx"]
    ny = grid_config["ny"]
    nz = grid_config["nz"]
    voxel_m = grid_config["voxel_size"]

    # Try .npy first (saved by notebook 04), fall back to raw .bin
    npy_path = Path("outputs/meshes") / f"{part_name}_density.npy"
    bin_path = Path("outputs/problem/density.bin")

    if npy_path.exists():
        density = np.load(npy_path).astype(np.float32)  # (nz, ny, nx)
        print(f"Loaded density from {npy_path}")
    elif bin_path.exists():
        density = np.fromfile(bin_path, dtype=np.float32).reshape(nz, ny, nx)
        print(f"Loaded density from {bin_path}")
    else:
        raise FileNotFoundError(
            f"Cannot find density: {npy_path} or {bin_path}\n"
            "Run notebook 04 first."
        )

    assert density.shape == (nz, ny, nx), \
        f"Expected ({nz},{ny},{nx}), got {density.shape}"

    rho = density.ravel()
    frac_solid = (rho > 0.9).mean()
    frac_void  = (rho < 0.1).mean()
    frac_grey  = 1.0 - frac_solid - frac_void

    print(f"\nDensity stats:")
    print(f"  Shape:        {density.shape}  [{nx*voxel_m*1000:.0f}×{ny*voxel_m*1000:.0f}×{nz*voxel_m*1000:.0f}mm]")
    print(f"  Min/Max/Mean: {rho.min():.4f} / {rho.max():.4f} / {rho.mean():.4f}")
    print(f"  ρ > 0.9 (solid): {frac_solid*100:.1f}%")
    print(f"  ρ < 0.1 (void):  {frac_void*100:.1f}%")
    print(f"  Grey (0.1–0.9):  {frac_grey*100:.1f}%   ← target: < 20%")

    if frac_grey > 0.30:
        print("\n  ⚠ WARNING: >30% grey elements — density not well-polarized.")
        print("    Re-run notebook 04 with two-stage penal continuation (p=2 → p=3).")
    elif frac_grey > 0.20:
        print("\n  ℹ NOTE: 20-30% grey — moderate polarization. Results usable.")
    else:
        print("\n  ✓ Polarization looks good.")

    # Histogram
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(rho, bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[0].axvline(DENSITY_THRESHOLD, color='red', linestyle='--',
                    label=f'threshold={DENSITY_THRESHOLD}')
    axes[0].set_xlabel('Density ρ')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Density histogram — target: bimodal peaks at 0 and 1')
    axes[0].legend()

    # Mid-Z slice
    iz_mid = nz // 2
    im = axes[1].imshow(density[iz_mid], vmin=0, vmax=1, cmap='bone',
                        origin='lower', aspect='auto')
    axes[1].set_title(f'Mid-Z slice (iz={iz_mid})')
    axes[1].set_xlabel('ix (x-axis)')
    axes[1].set_ylabel('iy (y-axis)')
    plt.colorbar(im, ax=axes[1])

    plt.tight_layout()
    density_hist_path = Path('outputs/reports') / f'{part_name}_density_histogram.png'
    plt.savefig(density_hist_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\nHistogram saved: {density_hist_path}")

else:
    # FEniCSx path — load from XDMF
    import h5py
    from mpi4py import MPI
    from dolfinx.io import XDMFFile

    xdmf_path    = Path("outputs/meshes/opt_domain.xdmf")
    density_path = Path(handoff.get('density_path',
                        f'outputs/meshes/{part_name}_density.xdmf'))
    assert xdmf_path.exists(), f"Not found: {xdmf_path}"
    assert density_path.exists(), f"Not found: {density_path}"

    comm = MPI.COMM_WORLD
    with XDMFFile(comm, str(xdmf_path), "r") as xdmf:
        domain = xdmf.read_mesh(name="Grid")
    coord_max = domain.geometry.x.max()
    if coord_max > 1.0:
        domain.geometry.x[:] /= 1000.0

    h5_path = str(density_path).replace('.xdmf', '.h5')
    with h5py.File(h5_path, 'r') as f:
        rho_values = f['Function/density/0'][:].ravel()

    pts_mm = domain.geometry.x * 1000.0
    tdim = domain.topology.dim
    domain.topology.create_connectivity(tdim, 0)
    conn   = domain.topology.connectivity(tdim, 0)
    n_elem = domain.topology.index_map(tdim).size_local
    tets   = np.array([conn.links(i) for i in range(n_elem)])
    print(f"FEniCSx mesh: {len(pts_mm):,} nodes, {n_elem:,} elements")
    print(f"Density range: [{rho_values.min():.4f}, {rho_values.max():.4f}]")


Loaded density from outputs/meshes/motor_mount_density.npy

Density stats:
  Shape:        (80, 60, 70)  [70×60×80mm]
  Min/Max/Mean: 0.0000 / 1.0000 / 0.2441
  ρ > 0.9 (solid): 22.7%
  ρ < 0.1 (void):  73.5%
  Grey (0.1–0.9):  3.8%   ← target: < 20%

  ✓ Polarization looks good.

Histogram saved: outputs/reports/motor_mount_density_histogram.png


Cell 3 — Isosurface extraction

In [4]:
# Cell 3 — Extract isosurface
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: check face counts and surface area before continuing.
# Raw MC face count for 100×60×20 grid at threshold 0.3 is typically
# 200k-600k. If you see < 5k faces the threshold is too high.
# ──────────────────────────────────────────────────────────────────────────
import trimesh
import numpy as np

if solver_type == 'rust_voxel':
    from skimage.measure import marching_cubes

    vs = voxel_m  # voxel_size in metres

    # Pad with one layer of zeros on all sides.
    # Without this, MC leaves open boundary loops wherever the density array
    # terminates — the top/bottom faces and hole cylinders never close.
    # The zero padding forces MC to generate a complete closed surface.
    density_padded = np.pad(density, pad_width=1, mode='constant', constant_values=0.0)

    print(f"Running marching cubes: threshold={DENSITY_THRESHOLD}, "
          f"spacing=({vs},{vs},{vs}) m  (padded shape: {density_padded.shape})")

    verts, faces, normals, _ = marching_cubes(
        density_padded,
        level=DENSITY_THRESHOLD,
        spacing=(vs, vs, vs),
        allow_degenerate=False,
    )

    # The padding shifted all verts by one voxel — subtract it back
    verts = verts - vs

    # Convert metres → mm for STL export (Fusion 360 expects mm)
    verts_mm = verts * 1000.0

    print(f"  Raw MC:  {len(verts):,} verts, {len(faces):,} faces")
    print(f"  Bounds:  {verts_mm.min(axis=0)} → {verts_mm.max(axis=0)} mm")

    mesh_raw = trimesh.Trimesh(
        vertices=verts_mm,
        faces=faces,
        vertex_normals=normals,
        process=True,   # degenerate face removal, duplicate vertex merge
    )
    print(f"  After process=True: {len(mesh_raw.vertices):,} verts, "
          f"{len(mesh_raw.faces):,} faces")
    print(f"  Watertight (raw):   {mesh_raw.is_watertight}")

else:
    # FEniCSx tet boundary extraction (kept from original skeleton)
    print(f"Building tet boundary mesh at threshold={DENSITY_THRESHOLD}")
    solid_mask = rho_values > DENSITY_THRESHOLD
    solid_tets = tets[solid_mask]
    TET_FACE_COMBOS = [[0,1,2],[0,1,3],[0,2,3],[1,2,3]]
    TET_OPP_VERTEX  = [3, 2, 1, 0]
    all_faces_raw = solid_tets[:, TET_FACE_COMBOS].reshape(-1, 3)
    sorted_faces  = np.sort(all_faces_raw, axis=1)
    _, inverse, counts = np.unique(sorted_faces, axis=0,
                                   return_inverse=True, return_counts=True)
    boundary_mask = counts[inverse] == 1
    boundary_faces_final = []
    seen_keys = set()
    for fi_flat, is_bnd in enumerate(boundary_mask):
        if not is_bnd: continue
        tet_idx  = fi_flat // 4
        face_idx = fi_flat % 4
        tet = solid_tets[tet_idx]
        fc  = TET_FACE_COMBOS[face_idx]
        key = tuple(sorted_faces[fi_flat])
        if key in seen_keys: continue
        seen_keys.add(key)
        v0,v1,v2 = pts_mm[tet[fc[0]]],pts_mm[tet[fc[1]]],pts_mm[tet[fc[2]]]
        opp_v  = pts_mm[tet[TET_OPP_VERTEX[face_idx]]]
        normal = np.cross(v1-v0, v2-v0)
        if np.dot(normal, (v0+v1+v2)/3.0 - opp_v) > 0:
            boundary_faces_final.append([tet[fc[0]], tet[fc[1]], tet[fc[2]]])
        else:
            boundary_faces_final.append([tet[fc[0]], tet[fc[2]], tet[fc[1]]])
    boundary_faces_final = np.array(boundary_faces_final)
    mesh_raw = trimesh.Trimesh(
        vertices=pts_mm,
        faces=boundary_faces_final,
        process=True,
    )
    print(f"  Tet boundary: {len(mesh_raw.vertices):,} verts, "
          f"{len(mesh_raw.faces):,} faces")
    print(f"  Watertight (raw): {mesh_raw.is_watertight}")

Running marching cubes: threshold=0.45, spacing=(0.001,0.001,0.001) m  (padded shape: (82, 62, 72))
  Raw MC:  43,083 verts, 86,258 faces
  Bounds:  [ 0.24841842  1.5725592  -0.5500001 ] → [77.55233 58.58509 69.55   ] mm
  After process=True: 43,083 verts, 86,258 faces
  Watertight (raw):   True


Cell 4 — Repair: largest component, fill holes, fix normals

In [5]:
# Cell 4 — Mesh repair
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: island count, open edge count, watertight status.
# A well-polarized density field should give 1-3 components and few open edges.
# Many small floating islands → threshold too low, or density not polarized.
# ──────────────────────────────────────────────────────────────────────────
import numpy as np

mesh = mesh_raw

def open_edge_count(m):
    return len(trimesh.grouping.group_rows(m.edges_sorted, require_count=1))

print(f"Before repair:  {len(mesh.vertices):,} verts, "
      f"{len(mesh.faces):,} faces, "
      f"{open_edge_count(mesh)} open edges")

# ── Step 1: Remove islands ────────────────────────────────────────────────
if REMOVE_ISLANDS:
    components = mesh.split(only_watertight=False)
    vols = [abs(c.volume) for c in components]
    total_vol = max(sum(vols), 1.0)
    large = [c for c, v in zip(components, vols)
             if v >= ISLAND_VOL_FRAC * total_vol]
    if len(large) == 0:
        large = [max(components, key=lambda m: len(m.faces))]
    n_dropped = len(components) - len(large)
    if n_dropped > 0:
        dropped_vol = sum(v for c,v in zip(components,vols)
                         if v < ISLAND_VOL_FRAC * total_vol)
        print(f"  Islands: dropped {n_dropped} components "
              f"({dropped_vol:.1f} mm³ total), kept {len(large)}")
    else:
        print(f"  No islands to drop ({len(components)} components, "
              f"all \u2265{ISLAND_VOL_FRAC*100:.0f}% of total volume)")
    mesh = trimesh.util.concatenate(large) if len(large) > 1 else large[0]

# ── Step 2: Merge duplicate vertices ─────────────────────────────────────
mesh.merge_vertices()

# ── Step 3: Fill holes ────────────────────────────────────────────────────
trimesh.repair.fill_holes(mesh)
trimesh.repair.fill_holes(mesh)  # second pass catches holes exposed by first
trimesh.repair.fix_normals(mesh)

# If still not watertight, voxel remesh as last resort
# (guarantees closure at the cost of slightly softened edges)
if not mesh.is_watertight:
    print("  Not watertight after fill_holes — re-extracting at higher threshold...")
    from skimage.measure import marching_cubes
    fallback_threshold = DENSITY_THRESHOLD + 0.10
    verts2, faces2, normals2, _ = marching_cubes(
        density,
        level=fallback_threshold,
        spacing=(voxel_m, voxel_m, voxel_m),
        allow_degenerate=False,
    )
    mesh2 = trimesh.Trimesh(
        vertices=verts2 * 1000.0,
        faces=faces2,
        process=True,
    )
    trimesh.repair.fill_holes(mesh2)
    trimesh.repair.fill_holes(mesh2)
    trimesh.repair.fix_normals(mesh2)
    if mesh2.is_watertight:
        print(f"  Fallback threshold {fallback_threshold:.2f} gave watertight mesh "
            f"({len(mesh2.faces):,} faces)")
        mesh = mesh2
        DENSITY_THRESHOLD = fallback_threshold  # propagate to downstream cells
    else:
        print(f"  Fallback threshold {fallback_threshold:.2f} still not watertight "
              f"— {open_edge_count(mesh2)} open edges")

# ── Step 4: Fix normals / winding ─────────────────────────────────────────
trimesh.repair.fix_normals(mesh)

# ── Step 5: Drop degenerate/duplicate faces ───────────────────────────────
mesh.update_faces(mesh.nondegenerate_faces())
mesh.update_faces(mesh.unique_faces())
mesh.remove_unreferenced_vertices()

oe = open_edge_count(mesh)
print(f"\nAfter repair:   {len(mesh.vertices):,} verts, "
      f"{len(mesh.faces):,} faces, "
      f"{oe} open edges")
print(f"Watertight:     {mesh.is_watertight}")
print(f"Winding consistent: {mesh.is_winding_consistent}")
if oe > 0:
    print(f"\n  \u26a0 {oe} open edges remain after repair.")
    print("    Options: increase DENSITY_THRESHOLD slightly, or")
    print("    re-run notebook 04 with two-stage continuation.")

Before repair:  43,083 verts, 86,258 faces, 0 open edges
  Islands: dropped 8 components (162.1 mm³ total), kept 1

After repair:   42,611 verts, 85,338 faces, 0 open edges
Watertight:     True
Winding consistent: True


Cell 5 — Taubin smoothing and quadric decimation

In [6]:
# Cell 5 — Smoothing and simplification
# ──────────────────────────────────────────────────────────────────────────
# CHECKPOINT: verify watertight status is preserved after each step.
# Taubin is volume-preserving; Laplacian alone causes shrinkage on curved surfaces.
# ──────────────────────────────────────────────────────────────────────────

# ── Taubin smoothing ─────────────────────────────────────────────────────
if SMOOTH_ITERATIONS > 0:
    vol_before = abs(mesh.volume) if mesh.is_watertight else None
    trimesh.smoothing.filter_taubin(
        mesh,
        lamb=TAUBIN_LAMB,
        nu=TAUBIN_NU,
        iterations=SMOOTH_ITERATIONS,
    )
    trimesh.repair.fix_normals(mesh)
    oe_after = open_edge_count(mesh)
    vol_after = abs(mesh.volume) if mesh.is_watertight else None
    vol_change = ((vol_after - vol_before) / vol_before * 100
                  if vol_before and vol_after else None)
    print(f"After Taubin ×{SMOOTH_ITERATIONS}: "
          f"{len(mesh.vertices):,} verts, "
          f"{len(mesh.faces):,} faces, "
          f"{oe_after} open edges")
    if vol_change is not None:
        print(f"  Volume: {vol_before:.1f} → {vol_after:.1f} mm³ "
              f"(Δ={vol_change:+.2f}%)")
    if vol_change is not None and abs(vol_change) > 5.0:
        print("  ⚠ >5% volume change — check TAUBIN_NU sign")

# ── Quadric decimation ────────────────────────────────────────────────────
n_raw = len(mesh.faces)
target_faces = max(int(n_raw * TARGET_FACE_FRACTION), MIN_FACES)
if target_faces < n_raw:
    print(f"\nDecimating {n_raw:,} → {target_faces:,} faces "
          f"({TARGET_FACE_FRACTION*100:.0f}%)...")
    mesh = mesh.simplify_quadric_decimation(target_faces)
    # One more repair pass after decimation can introduce new holes
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    mesh.remove_unreferenced_vertices()
    oe_final = open_edge_count(mesh)
    print(f"After decimation: {len(mesh.vertices):,} verts, "
          f"{len(mesh.faces):,} faces, "
          f"{oe_final} open edges")
else:
    print(f"\nSkipping decimation (face count {n_raw:,} already ≤ target {target_faces:,})")

print(f"\nFinal mesh:")
print(f"  Verts:      {len(mesh.vertices):,}")
print(f"  Faces:      {len(mesh.faces):,}")
print(f"  Watertight: {mesh.is_watertight}")
print(f"  Winding:    {mesh.is_winding_consistent}")
if mesh.is_watertight:
    vol = abs(mesh.volume)
    # Expected: vf * domain_volume (mm³)
    if solver_type == 'rust_voxel':
        domain_vol = (voxel_m * nx * 1000) * (voxel_m * ny * 1000) * (voxel_m * nz * 1000)
        expected_vol = handoff.get('final_volume_frac', 0.45) * domain_vol
        print(f"  Volume:     {vol:.1f} mm³  (expected ≈{expected_vol:.0f} mm³)")
    else:
        print(f"  Volume:     {vol:.1f} mm³")


After Taubin ×10: 42,611 verts, 85,338 faces, 0 open edges
  Volume: 83050.8 → 81500.4 mm³ (Δ=-1.87%)

Decimating 85,338 → 25,601 faces (30%)...
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
After decimation: 12,742 verts, 25,600 faces, 0 open edges

Final mesh:
  Verts:      12,742
  Faces:      25,600
  Watertight: True
  Winding:    True
  Volume:     81489.7 mm³  (expected ≈84000 mm³)


Cell 6 — Manifold verification and STL export

In [7]:
# Cell 6 — Final verification and STL export
from pathlib import Path

stl_dir = Path("outputs/stl")
stl_dir.mkdir(parents=True, exist_ok=True)
stl_path = stl_dir / f"{part_name}_optimized.stl"

# Final manifold check
issues = []
if not mesh.is_watertight:         issues.append("not watertight")
if not mesh.is_winding_consistent: issues.append("winding inconsistent")
oe = open_edge_count(mesh)
if oe > 0:                          issues.append(f"{oe} open edges")

if issues:
    print(f"⚠ Quality issues: {', '.join(issues)}")
    print("  The STL will still export but may need repair in Fusion 360.")
    print("  Suggestions:")
    print("    - Increase DENSITY_THRESHOLD (try +0.05)")
    print("    - Re-run Stage 4 with two-stage penal continuation")
    print("    - Increase SMOOTH_ITERATIONS (try 50)")
else:
    print("✓ All quality checks passed")

mesh.export(str(stl_path))
stl_size_kb = stl_path.stat().st_size / 1024

print(f"\nSTL exported:")
print(f"  Path:       {stl_path}")
print(f"  Size:       {stl_size_kb:.1f} KB")
print(f"  Triangles:  {len(mesh.faces):,}")
if mesh.is_watertight:
    print(f"  Volume:     {abs(mesh.volume):.2f} mm³")
    print(f"  Bounds:     {mesh.bounds[0]} → {mesh.bounds[1]} mm")

# Quality checklist
print("\nQuality checklist (Fusion 360 import):")
print(f"  [{'✓' if mesh.is_watertight else '✗'}] is_watertight")
print(f"  [{'✓' if mesh.is_winding_consistent else '✗'}] is_winding_consistent")
print(f"  [{'✓' if oe == 0 else '✗'}] 0 open edges")
print(f"  [{'✓' if len(mesh.faces) < 100_000 else '⚠'}] triangles < 100k")
if mesh.is_watertight:
    vol = abs(mesh.volume)
    vf_check = 0.45
    if solver_type == 'rust_voxel':
        domain_vol = (voxel_m * nx * 1000) * (voxel_m * ny * 1000) * (voxel_m * nz * 1000)
        vf_mesh = vol / domain_vol
        vf_target = handoff.get('final_volume_frac', 0.45)
        ok = abs(vf_mesh - vf_target) < 0.10
        print(f"  [{'✓' if ok else '⚠'}] volume fraction ≈ target "
              f"(mesh: {vf_mesh:.3f}, target: {vf_target:.3f})")


✓ All quality checks passed

STL exported:
  Path:       outputs/stl/motor_mount_optimized.stl
  Size:       1250.1 KB
  Triangles:  25,600
  Volume:     81489.75 mm³
  Bounds:     [ 0.38712317  2.04307986 -0.52651211] → [77.43261604 58.40713917 69.54907534] mm

Quality checklist (Fusion 360 import):
  [✓] is_watertight
  [✓] is_winding_consistent
  [✓] 0 open edges
  [✓] triangles < 100k
  [✓] volume fraction ≈ target (mesh: 0.243, target: 0.250)


Cell 6b — Boolean hole cutting (DISABLED)

In [8]:
# Cell 7 — Before/after render
if not RENDER_PLOTS:
    print("Skipping render — set RENDER_PLOTS=True after fixing display")
else:
    import pyvista as pv
    pv.OFF_SCREEN = True

    report_dir = Path('outputs/reports')
    original_stl = Path(f'outputs/meshes/{part_name}.stl')

    pl = pv.Plotter(shape=(1, 2), window_size=(1600, 600))
    if original_stl.exists():
        orig = pv.read(str(original_stl))
        pl.subplot(0, 0)
        pl.add_mesh(orig, color='lightsteelblue', show_edges=False, opacity=0.9)
        pl.add_text(f'Original\n{orig.n_cells:,} faces', font_size=10)
        pl.view_isometric()
    opt = pv.read(str(stl_path))
    pl.subplot(0, 1)
    pl.add_mesh(opt, color='coral', show_edges=False, opacity=0.9)
    pl.add_text(f'Optimized\n{opt.n_cells:,} faces', font_size=10)
    pl.view_isometric()
    compare_png = report_dir / f'{part_name}_before_after.png'
    pl.screenshot(str(compare_png))
    pl.close()
    print(f'Before/after: {compare_png}')


Skipping render — set RENDER_PLOTS=True after fixing display


Cell 8 — Pipeline completion record

In [9]:
# Cell 8 — Write stage05 completion record
import json
from pathlib import Path
from datetime import datetime, timezone

stage_handoffs = {}
for stage_json in sorted(Path('outputs/meshes').glob(f'{part_name}_stage0*.json')):
    try:
        d = json.loads(stage_json.read_text())
        stage_handoffs[d['stage']] = d
    except Exception:
        pass

_s04 = stage_handoffs.get('04_simp', handoff)

record = {
    "stage":          "05_stl_export",
    "part_name":      part_name,
    "completed_at":   datetime.now(timezone.utc).isoformat(),
    "stl_path":       str(stl_path),
    "stl_size_kb":    round(stl_size_kb, 2),
    "mesh": {
        "vertices":   len(mesh.vertices),
        "triangles":  len(mesh.faces),
        "watertight": mesh.is_watertight,
        "volume_mm3": round(abs(mesh.volume), 3),
        "area_mm2":   round(mesh.area, 3),
    },
    'export_config': {
        'density_threshold':   DENSITY_THRESHOLD,
        'smooth_iterations':   SMOOTH_ITERATIONS,
        'taubin_lamb':         TAUBIN_LAMB,
        'taubin_nu':           TAUBIN_NU,
        'target_face_fraction': TARGET_FACE_FRACTION,
        'remove_islands':      REMOVE_ISLANDS,
    },
    'optimization': {
        'solver':          _s04.get('solver'),
        'volume_fraction': _s04.get('config', {}).get('volume_fraction'),
        'n_iterations':    _s04.get('n_iterations'),
        'converged':       _s04.get('converged'),
        'final_compliance':_s04.get('final_compliance'),
    },
}

record_path = Path('outputs/meshes') / f'{part_name}_stage05.json'
record_path.write_text(json.dumps(record, indent=2))

print('═' * 60)
print('  PIPELINE COMPLETE')
print('═' * 60)
print(f'  Part:          {part_name}')
print(f'  STL:           {stl_path}')
print(f'  Watertight:    {mesh.is_watertight}')
print(f'  Triangles:     {len(mesh.faces):,}')
if mesh.is_watertight:
    print(f'  Volume:        {abs(mesh.volume):.2f} mm³')
_vf = _s04.get('config', {}).get('volume_fraction')
if _vf:
    print(f'  Material saved: {(1 - _vf) * 100:.1f}%')
print('─' * 60)
print('  Stage durations:')
for stage, data in sorted(stage_handoffs.items()):
    dur = data.get('duration_s', '—')
    print(f'    {stage:<20} {dur}s')
print('═' * 60)
print(f'\n  Full record: {record_path}')


════════════════════════════════════════════════════════════
  PIPELINE COMPLETE
════════════════════════════════════════════════════════════
  Part:          motor_mount
  STL:           outputs/stl/motor_mount_optimized.stl
  Watertight:    True
  Triangles:     25,600
  Volume:        81489.75 mm³
  Material saved: 75.0%
────────────────────────────────────────────────────────────
  Stage durations:
    01_geometry          0.644s
    02_meshing           —s
    03_fea               —s
    04_simp              24099.64035582599s
    05_stl_export        —s
════════════════════════════════════════════════════════════

  Full record: outputs/meshes/motor_mount_stage05.json
